7) 벡터 스토어 검색 메모리

In [ ]:
# [목적] 대화 내용을 벡터로 저장하고 유사한 기록을 찾을 FAISS 벡터 저장소를 준비하는 예제
# 텍스트를 임베딩 벡터로 변환하는 모델과, 벡터 검색을 수행할 인메모리 FAISS 저장소를 연결합니다.
# 이후 질문과 의미가 가까운 과거 대화만 찾아 메모리 문맥으로 사용하기 위한 기반입니다.
import faiss
from langchain_openai import OpenAIEmbeddings
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv

load_dotenv()

embeddings_model = OpenAIEmbeddings()  # 임베딩 모델을 정의

embedding_size = 1536  # Vector Store를 초기화
# IndexFlatL2는 벡터 사이의 거리를 계산해 의미가 가까운 기록을 찾는 FAISS 인덱스입니다.
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(embeddings_model, index, InMemoryDocstore({}), {})

In [ ]:
# [목적] 벡터 저장소에서 관련 대화를 찾아오는 VectorStoreRetrieverMemory를 구성하는 예제
# retriever가 새 질문과 가장 유사한 기록을 검색하고, 메모리가 그 결과를 대화 문맥으로 제공합니다.
# 전체 대화가 아닌 필요한 정보만 모델에 전달해 긴 대화를 효율적으로 처리합니다.
from langchain_classic.memory import VectorStoreRetrieverMemory

# k=1은 질문과 가장 유사한 대화 기록 한 개를 검색한다는 뜻입니다.
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

memory = VectorStoreRetrieverMemory(retriever=retriever)

In [ ]:
# [목적] 여러 자기소개 문답을 벡터 저장소 기반 메모리에 저장하는 예제
# save_context가 각 문답을 임베딩으로 변환해 저장하므로, 나중에 의미가 비슷한 질문으로 검색할 수 있습니다.
# 프로젝트 경험과 강점처럼 질문마다 필요한 과거 정보를 선택적으로 꺼내기 위해 대화를 축적합니다.
memory.save_context(  # 임의의 대화를 저장
    inputs={"human": "안녕하세요, 오늘 면접에 ... 자기소개 부탁드립니다."},
    outputs={
        "ai": "안녕하세요. 저는 컴퓨터 과학을 전공한 신입 개발자입니다. 대학에서는 주로 자바와 파이썬을 사용했으며, 최근에는 웹 개발 프로젝트에 참여하여 실제 사용자를 위한 서비스를 개발하는 경험을 했습니다."
    },
)

memory.save_context(
    inputs={"human": "프로젝트에서 어떤 역할을 맡았나요?"},
    outputs={
        "ai": "제가 맡은 역할은 백엔드 개발자였습니다. 사용자 데이터 처리와 서버 로직 개발을 담당했으며, RESTful API를 구현하여 프론트엔드와의 통신을 담당했습니다. 또한, 데이터베이스 설계에도 참여했습니다."
    },
)

memory.save_context(
    inputs={"human": "팀 프로젝트에서 어려움을 겪었던 경험이 있다면 ..."},
    outputs={
        "ai": "프로젝트 초기에 의사소통 문제로 몇 가지 어려움이 ..."
    },
)

memory.save_context(
    inputs={"human": "개발자로서 자신의 강점은 무엇이라고 생각하나요?"},
    outputs={
        "ai": "제 강점은 빠른 학습 능력과 문제 해결 능력입니다. ..."
    },
)

In [ ]:
# [목적] 새 질문과 의미가 가까운 과거 대화를 검색해 확인하는 예제
# load_memory_variables가 질문을 벡터로 바꾼 뒤 가장 유사한 저장 기록을 history로 반환합니다.
# 검색된 결과가 답변에 제공할 관련 문맥으로 적절한지 확인합니다.ㄴ
print(memory.load_memory_variables({"human": "면접자 전공은 무엇인가요?"})["history"])

In [ ]:
# [목적] 다른 주제의 질문으로 벡터 메모리가 관련 프로젝트 경험을 찾는지 확인하는 예제
# 질문의 의미와 가까운 문답이 저장소에서 검색되어 history에 반환됩니다.
# 키워드가 달라도 의미가 유사하면 과거 정보를 활용할 수 있음을 보여 줍니다.
print(
    memory.load_memory_variables(
        {"human": "면접자가 프로젝트에서 맡은 역할은 무엇인가요?"}
    )["history"]
)